# Линейная регрессия для концентрации CO2

Этот notebook повторяет логику `analysis.py`, но оформлен как пошаговый Jupyter-проект.
Он загружает данные Mauna Loa CO2, обучает линейную регрессию с нуля, считает метрики и сохраняет графики в папку `output_plots/`.


## 1. Импорт библиотек и настройка

В этом блоке подключаем библиотеки, задаём параметры эксперимента и оформляем стиль графиков.


In [ ]:
import urllib.request
import io
import os
import math

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

DATASET_URL = "https://raw.githubusercontent.com/datasets/co2-ppm/master/data/co2-mm-mlo.csv"
OUTPUT_DIR = "output_plots"
TRAIN_SPLIT = 0.8

DARK_BG = "#0f1117"
PANEL_BG = "#1a1d27"
ACCENT1 = "#4fc3f7"
ACCENT2 = "#f06292"
ACCENT3 = "#81c784"
GRID_COL = "#2e3248"
TEXT_COL = "#e0e0e0"
WARN_COL = "#ffd54f"

plt.rcParams.update({
    "figure.facecolor": DARK_BG,
    "axes.facecolor": PANEL_BG,
    "axes.edgecolor": GRID_COL,
    "axes.labelcolor": TEXT_COL,
    "axes.titlecolor": TEXT_COL,
    "xtick.color": TEXT_COL,
    "ytick.color": TEXT_COL,
    "grid.color": GRID_COL,
    "grid.linewidth": 0.6,
    "text.color": TEXT_COL,
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})


## 2. Загрузка данных и функции расчёта

Здесь находятся функции для скачивания датасета, ручного расчёта статистик и разбиения на train/test.


In [ ]:
def load_data(url: str) -> pd.DataFrame:
    """Скачивает CSV и возвращает очищенный DataFrame."""
    print(f"[+] Загружаю датасет из:\n    {url}\n")
    with urllib.request.urlopen(url) as response:
        raw = response.read().decode("utf-8")

    cols = ["Date", "Decimal_Date", "Average", "Interpolated",
            "Seasonally_Adjusted", "Trend", "Num_Days"]
    df = pd.read_csv(io.StringIO(raw), names=cols, skiprows=1)

    for col in ["Decimal_Date", "Average", "Interpolated"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[df["Average"] > 0].reset_index(drop=True)
    print(f"[+] Загружено {len(df)} ежемесячных наблюдений"
          f" ({df['Date'].iloc[0]} -> {df['Date'].iloc[-1]})\n")
    return df


def mean(data: list) -> float:
    return sum(data) / len(data)


def std(data: list) -> float:
    mu = mean(data)
    return math.sqrt(sum((x - mu) ** 2 for x in data) / len(data))


def covariance(x: list, y: list) -> float:
    mx, my = mean(x), mean(y)
    return sum((xi - mx) * (yi - my) for xi, yi in zip(x, y)) / len(x)


def correlation(x: list, y: list) -> float:
    denom = std(x) * std(y)
    return covariance(x, y) / denom if denom != 0 else 0.0


def linear_regression(x: list, y: list) -> tuple[float, float]:
    """Обычный МНК: возвращает (наклон, свободный член)."""
    sx, sy = std(x), std(y)
    if sx == 0:
        return 0.0, mean(y)
    slope = covariance(x, y) / (sx ** 2)
    intercept = mean(y) - slope * mean(x)
    return slope, intercept


def predict(x: list, slope: float, intercept: float) -> list:
    return [slope * xi + intercept for xi in x]


def mae(actual: list, predicted: list) -> float:
    return sum(abs(a - p) for a, p in zip(actual, predicted)) / len(actual)


def mse(actual: list, predicted: list) -> float:
    return sum((a - p) ** 2 for a, p in zip(actual, predicted)) / len(actual)


def rmse(actual: list, predicted: list) -> float:
    return math.sqrt(mse(actual, predicted))


def r_squared(actual: list, predicted: list) -> float:
    mu = mean(actual)
    ss_res = sum((a - p) ** 2 for a, p in zip(actual, predicted))
    ss_tot = sum((a - mu) ** 2 for a in actual)
    return 1 - ss_res / ss_tot if ss_tot != 0 else 0.0


def train_test_split(df: pd.DataFrame, split: float):
    n = int(len(df) * split)
    return df.iloc[:n].copy(), df.iloc[n:].copy()


def compute_residuals(actual: list, predicted: list) -> list:
    return [a - p for a, p in zip(actual, predicted)]


def save(fig, name: str):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
    plt.close(fig)
    print(f"    Сохранено -> {path}")


## 3. Построение графиков

Этот блок отвечает за все визуализации: исходный ряд, регрессию, сравнение факта и прогноза, остатки, сводку метрик и экстраполяцию в будущее.


In [ ]:
def plot_full_series(df: pd.DataFrame):
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(df["Decimal_Date"], df["Average"], color=ACCENT1, lw=0.9, alpha=0.85)
    ax.fill_between(df["Decimal_Date"], df["Average"], df["Average"].min() - 2,
                    alpha=0.12, color=ACCENT1)
    ax.set_title("Концентрация CO2 в атмосфере — обсерватория Мауна-Лоа")
    ax.set_xlabel("Год")
    ax.set_ylabel("CO2 (ppm)")
    ax.grid(True, axis="y")
    last_val = df["Average"].iloc[-1]
    last_yr = df["Decimal_Date"].iloc[-1]
    ax.annotate(f"{last_val:.1f} ppm",
                xy=(last_yr, last_val),
                xytext=(-80, 15),
                textcoords="offset points",
                color=WARN_COL,
                fontsize=10,
                arrowprops=dict(arrowstyle="->", color=WARN_COL, lw=1.2))
    save(fig, "01_full_series.png")


def plot_regression(df_train, df_test, slope, intercept):
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.scatter(df_train["Decimal_Date"], df_train["Average"],
               s=3, color=ACCENT1, alpha=0.55, label="Обучающая выборка")
    ax.scatter(df_test["Decimal_Date"], df_test["Average"],
               s=5, color=ACCENT2, alpha=0.75, label="Тестовая выборка", zorder=3)
    x_range = [df_train["Decimal_Date"].min(), df_test["Decimal_Date"].max()]
    y_range = [slope * x + intercept for x in x_range]
    ax.plot(x_range, y_range, color=ACCENT3, lw=2.2,
            label=f"Регрессия  y = {slope:.4f}x + {intercept:.1f}")
    ax.set_title("Линейная регрессия: CO2 во времени")
    ax.set_xlabel("Год")
    ax.set_ylabel("CO2 (ppm)")
    ax.legend(framealpha=0.15)
    ax.grid(True)
    save(fig, "02_regression_line.png")


def plot_predictions_vs_actual(df_test, test_pred):
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(df_test["Decimal_Date"], df_test["Average"],
            color=ACCENT1, lw=1.5, label="Фактический CO2")
    ax.plot(df_test["Decimal_Date"], test_pred,
            color=ACCENT2, lw=1.5, linestyle="--", label="Прогноз (линейный)")
    ax.fill_between(df_test["Decimal_Date"], df_test["Average"], test_pred,
                    alpha=0.18, color=WARN_COL, label="Полоса ошибки")
    ax.set_title("Тестовый период — факт и прогноз CO2")
    ax.set_xlabel("Год")
    ax.set_ylabel("CO2 (ppm)")
    ax.legend(framealpha=0.15)
    ax.grid(True)
    save(fig, "03_actual_vs_predicted.png")


def plot_residuals(df_test, residuals):
    fig = plt.figure(figsize=(14, 6))
    gs = gridspec.GridSpec(1, 2, width_ratios=[2.5, 1], wspace=0.08)

    ax1 = fig.add_subplot(gs[0])
    ax1.scatter(df_test["Decimal_Date"], residuals,
                s=8, color=ACCENT2, alpha=0.65)
    ax1.axhline(0, color=ACCENT3, lw=1.5, linestyle="--")
    ax1.set_title("Остатки во времени (тестовая выборка)")
    ax1.set_xlabel("Год")
    ax1.set_ylabel("Остаток (ppm)")
    ax1.grid(True)

    ax2 = fig.add_subplot(gs[1], sharey=ax1)
    ax2.hist(residuals, bins=28, orientation="horizontal",
             color=ACCENT1, alpha=0.75, edgecolor=DARK_BG)
    ax2.axhline(0, color=ACCENT3, lw=1.5, linestyle="--")
    ax2.set_xlabel("Частота")
    ax2.set_title("Распределение")
    ax2.grid(True, axis="x")
    plt.setp(ax2.get_yticklabels(), visible=False)
    save(fig, "04_residuals.png")


def plot_metrics_dashboard(metrics: dict):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.axis("off")
    fig.patch.set_facecolor(PANEL_BG)

    ax.text(0.5, 0.92, "Сводка качества модели", ha="center", va="top",
            fontsize=16, fontweight="bold", color=TEXT_COL,
            transform=ax.transAxes)

    labels = {
        "Наклон (ppm/год)": f"{metrics['slope']:.4f}",
        "Свободный член": f"{metrics['intercept']:.2f}",
        "R2 (train)": f"{metrics['r2_train']:.6f}",
        "R2 (test)": f"{metrics['r2_test']:.6f}",
        "MAE (test, ppm)": f"{metrics['mae']:.4f}",
        "RMSE (test, ppm)": f"{metrics['rmse']:.4f}",
        "Корреляция (r)": f"{metrics['corr']:.6f}",
        "Наблюдений в train": str(metrics['n_train']),
        "Наблюдений в test": str(metrics['n_test']),
    }

    colors_v = [ACCENT3, ACCENT3, ACCENT1, ACCENT1,
                ACCENT2, ACCENT2, WARN_COL, TEXT_COL, TEXT_COL]
    y = 0.80
    for (label, value), color in zip(labels.items(), colors_v):
        ax.text(0.15, y, label, ha="left", va="top", fontsize=12,
                color=TEXT_COL, transform=ax.transAxes)
        ax.text(0.75, y, value, ha="left", va="top", fontsize=12,
                color=color, fontweight="bold", transform=ax.transAxes)
        ax.plot([0.05, 0.95], [y - 0.005, y - 0.005], color=GRID_COL,
                lw=0.5, transform=ax.transAxes, clip_on=False)
        y -= 0.088

    save(fig, "05_metrics_dashboard.png")


def plot_future_forecast(df: pd.DataFrame, slope: float, intercept: float,
                         years_ahead: int = 25):
    last_year = df["Decimal_Date"].max()
    future_x = [last_year + i * (1 / 12) for i in range(1, years_ahead * 12 + 1)]
    future_y = predict(future_x, slope, intercept)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(df["Decimal_Date"], df["Average"],
            color=ACCENT1, lw=0.9, alpha=0.7, label="Исторические данные")
    ax.plot(future_x, future_y,
            color=ACCENT2, lw=2.0, linestyle="--",
            label=f"Линейный прогноз (+{years_ahead} лет)")

    target_yr = last_year + years_ahead
    target_co2 = slope * target_yr + intercept
    ax.axvline(target_yr, color=WARN_COL, lw=1.0, linestyle=":")
    ax.text(target_yr + 0.3, target_co2 - 5,
            f"~{target_co2:.0f} ppm\nв {int(target_yr)} году",
            color=WARN_COL, fontsize=10)

    ax.set_title(f"Прогноз CO2 — линейная экстраполяция до {int(target_yr)} года")
    ax.set_xlabel("Год")
    ax.set_ylabel("CO2 (ppm)")
    ax.legend(framealpha=0.15)
    ax.grid(True)
    save(fig, "06_forecast.png")


## 4. Запуск анализа

В последнем блоке последовательно выполняются: загрузка данных, разбиение, обучение модели, расчёт метрик и создание графиков.


In [ ]:
print("=" * 60)
print("  Анализ линейной регрессии для CO2")
print("=" * 60)

df = load_data(DATASET_URL)
df_train, df_test = train_test_split(df, TRAIN_SPLIT)
print(f"[+] Train: {len(df_train)} строк  |  Test: {len(df_test)} строк\n")

x_train = df_train["Decimal_Date"].tolist()
y_train = df_train["Average"].tolist()
x_test = df_test["Decimal_Date"].tolist()
y_test = df_test["Average"].tolist()

slope, intercept = linear_regression(x_train, y_train)
print("[+] Параметры линейной регрессии:")
print(f"    slope     = {slope:.4f}  ppm / year")
print(f"    intercept = {intercept:.2f}\n")

train_pred = predict(x_train, slope, intercept)
test_pred = predict(x_test, slope, intercept)

r2_train = r_squared(y_train, train_pred)
r2_test = r_squared(y_test, test_pred)
mae_val = mae(y_test, test_pred)
rmse_val = rmse(y_test, test_pred)
corr_val = correlation(df["Decimal_Date"].tolist(), df["Average"].tolist())
residuals = compute_residuals(y_test, test_pred)

metrics = dict(
    slope=slope, intercept=intercept,
    r2_train=r2_train, r2_test=r2_test,
    mae=mae_val, rmse=rmse_val,
    corr=corr_val,
    n_train=len(x_train), n_test=len(x_test),
)

print("[+] Метрики:")
print(f"    R2 (train) = {r2_train:.6f}")
print(f"    R2 (test)  = {r2_test:.6f}")
print(f"    MAE        = {mae_val:.4f} ppm")
print(f"    RMSE       = {rmse_val:.4f} ppm")
print(f"    Correlation = {corr_val:.6f}\n")

print("[+] Генерирую графики...")
plot_full_series(df)
plot_regression(df_train, df_test, slope, intercept)
plot_predictions_vs_actual(df_test, test_pred)
plot_residuals(df_test, residuals)
plot_metrics_dashboard(metrics)
plot_future_forecast(df, slope, intercept, years_ahead=25)

print(f"\n[OK] Готово. Графики сохранены в ./{OUTPUT_DIR}/")
print("=" * 60)
